# Milestone 2 — Encoder trigger head (DeBERTa-v3-large)

First vertical slice of the Path B encoder parser: fine-tune
`microsoft/deberta-v3-large` as a **trigger-identification token classifier**,
then score it with the same word-level metric as the baseline.

**Baseline to beat (trigger id):** F1 `0.735`, and `196.6 ms/sample` — the
encoder does one forward pass per sentence, so watch the ms/sentence drop hard.

Same robustness rules as the reproduction notebook: **clone, don't pip-build**
our code; leave Colab's torch/protobuf/numpy alone; set the protobuf env var
before any import.

> `Runtime → GPU` (A100 recommended), then `Run all`. The train cell (step 4)
> mounts Google Drive so the model survives a disconnect.

In [ ]:
!nvidia-smi

## 1. Clone the project + install the pinned env

Clones the private `texturejc/Texture_Frames` repo — which contains both the
`encoder_parser/` code and the vendored `frame-semantic-transformer/` data
loaders — and installs `requirements-colab.txt`.

**One-time:** add a GitHub PAT (repo scope) to Colab **Secrets** (🔑 sidebar) as
`GH_TOKEN` with notebook access enabled.

In [ ]:
# --- Clone the private project repo -----------------------------------------
# One-time setup: add a GitHub PAT (repo scope) to Colab Secrets (🔑 left sidebar)
# named GH_TOKEN, with notebook access enabled.
import os
try:
    from google.colab import userdata
    os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN") or ""
except Exception:
    os.environ.setdefault("GH_TOKEN", "")  # or set manually: os.environ["GH_TOKEN"] = "ghp_..."

# $GH_TOKEN is read from the shell env, so the token never appears in the source.
# If Texture_Frames already exists, pull the latest instead of re-cloning.
![ -d Texture_Frames ] && (cd Texture_Frames && git pull -q) || git clone -q https://$GH_TOKEN@github.com/texturejc/Texture_Frames.git
!cd Texture_Frames && echo "project at $(git rev-parse --short HEAD)"

# Slim pinned env: transformers/tokenizers/sentencepiece/nltk/accelerate only.
# No datasets/pyarrow, no nlpaug — those caused the numpy-ABI crashes. torch /
# protobuf / numpy left as Colab ships them.
!pip install -q -r Texture_Frames/requirements-colab.txt

## 2. Environment + FrameNet data

Sets the protobuf env var (before any transformers import) and puts
`encoder_parser/` on `sys.path`. `data.py` finds the vendored parser at
`../frame-semantic-transformer` relative to its own location, so both resolve
straight from the clone — no manual uploads.

In [ ]:
import os, sys
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"  # before transformers
sys.path.insert(0, "/content/Texture_Frames/encoder_parser")

import nltk
for pkg in ["framenet_v17", "wordnet", "omw-1.4"]:
    nltk.download(pkg)

## 3. Sanity-check the data pipeline (cheap, catches alignment bugs early)

Before a multi-hour train, confirm the trigger labels line up with the text.

In [ ]:
from transformers import AutoTokenizer
import data

tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
print("fast tokenizer:", tok.is_fast)

raw = data.load_trigger_sentences("dev")
print(f"dev sentences: {len(raw)}")
text, locs = raw[0]
print("text :", text)
print("gold trigger words:", [text[s:e] for s, e in data.whitespace_words(text)
                              if s in {data.snap_to_word_start(text, l) for l in locs}])

# Inspect one tokenized example's labels.
ds = data.build_trigger_dataset("dev", tok, max_length=320)
ex = ds[0]
toks = tok.convert_ids_to_tokens(ex["input_ids"])
print("\ntoken -> label (only non -100 shown):")
for t, l in zip(toks, ex["labels"]):
    if l != -100:
        print(f"  {t!r:20} {data.ID2LABEL[l]}")

## 4. Train

~5 epochs over the FrameNet training docs. On an A100 expect roughly 15–40 min.

In [ ]:
import train_trigger

# Mount Drive so the trained model survives a disconnect (falls back to local).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/texture_frames/trigger"
except Exception:
    OUTPUT_DIR = "outputs/trigger"
print("saving to:", OUTPUT_DIR)

model, tokenizer = train_trigger.train(
    base_model="microsoft/deberta-v3-large",
    output_dir=OUTPUT_DIR,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    max_length=320,
)

## 5. Evaluate — word-level trigger F1 (baseline-comparable) + speed

In [ ]:
from eval_trigger import evaluate_trigger, print_report

metrics = evaluate_trigger(model, tokenizer, split="test", max_length=320)
print_report(metrics, reported_f1=0.735)

## Interpreting

This is the **scaffold** run — the goal is a working end-to-end pipeline and a
first honest F1, not necessarily beating 0.735 on attempt one. Two numbers to
report back:
1. **trigger F1** vs. 0.735, and
2. **ms/sentence** vs. the baseline's 196.6 ms/sample.

If F1 is in the ballpark (say >0.68) the pipeline is sound and Milestone 3 tuning
takes it the rest of the way. If it's near zero, it's almost always a
label-alignment bug — the Cell 3 sanity check is where we'd catch it.